In [53]:
import ee
import geemap

import gee_general_helpers as gee_g_help

GEE_PROJECT = "ee-tymc5571-multi-disturbance"
PROJECT_DIR = "projects/ee-tymc5571-multi-disturbance/assets"

# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------

YEARS = list(range(2000, 2025))   # 2000-2024 inclusive

FOREST_MASKS_ASSET = "projects/ee-tymc5571-cbi-module/assets/forest_masks"
WUMI_SEV_ASSET = 'projects/ee-tymc5571-cbi-module/assets/WUMI2024a_allsev'
FOREST_DIST_IC = f"{PROJECT_DIR}/dist_from_conservative_forest"
WUMI_CBI_MW_IC = f'{PROJECT_DIR}/WUMI_cbi_mw'


DIST_NEIGHBORHOOD = 100           # pixels
PIXEL_SIZE_M = 30
DIST_CAP_M = DIST_NEIGHBORHOOD * PIXEL_SIZE_M
MOVING_WINDOW_SIZE = 500

# Optional export settings
EXPORT_SCALE = 30
EXPORT_MAX_PIXELS = 1e13

# -----------------------------------------------------------------------------
# AUTH / INIT
# -----------------------------------------------------------------------------

ee.Authenticate(
    auth_mode="notebook",
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.read_write",
        "https://www.googleapis.com/auth/drive",
    ],
)
ee.Initialize(project=GEE_PROJECT)

In [54]:

# -----------------------------------------------------------------------------
# INPUTS
# -----------------------------------------------------------------------------

forest_masks = ee.ImageCollection(FOREST_MASKS_ASSET)
wumi_sev = ee.ImageCollection(WUMI_SEV_ASSET)

gee_g_help.ensure_imagecollection(FOREST_DIST_IC)
gee_g_help.ensure_imagecollection(WUMI_CBI_MW_IC)



ImageCollection already exists: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest
ImageCollection already exists: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw


# Distance from forest

In [26]:
# -----------------------------------------------------------------------------
# PREPARE ANNUAL CONSERVATIVE FOREST IMAGES
# -----------------------------------------------------------------------------

years_ee = ee.List(YEARS)

conservative_forest_ic = (
    forest_masks
    .select(["conservative"])
    .filter(ee.Filter.notNull(["year"]))
    .filter(ee.Filter.inList("year", years_ee))
    .sort("year")
)

def prep_conservative_forest(img):
    img = ee.Image(img)
    year = ee.Number(img.get("year"))

    out = (
        img
        .rename("conservative_forest")
        .set("year", year)
        .set("system:time_start", ee.Date.fromYMD(year, 1, 1).millis())
    )

    return out

conservative_forest_ic = ee.ImageCollection(
    conservative_forest_ic.map(prep_conservative_forest)
)

# -----------------------------------------------------------------------------
# COMPUTE DISTANCE FROM CONSERVATIVE FOREST
# -----------------------------------------------------------------------------

def compute_forest_distance(img):
    img = ee.Image(img)
    year = ee.Number(img.get("year"))

    dist_m = (
        img.fastDistanceTransform(
            neighborhood=DIST_NEIGHBORHOOD,
            units="pixels",
            metric="squared_euclidean",
        )
        .sqrt()
        .multiply(PIXEL_SIZE_M)
    )
    dist_m = dist_m.min(DIST_CAP_M)
    
    out = (
        dist_m
        .rename("dist_from_conservative_forest_m")
        .set("year", year)
        .set("distance_neighborhood_pixels", DIST_NEIGHBORHOOD)
        .set("system:time_start", ee.Date.fromYMD(year, 1, 1).millis())
    )

    return out

forest_distance_ic = ee.ImageCollection(
    conservative_forest_ic.map(compute_forest_distance)
)

# -----------------------------------------------------------------------------
# QUICK CHECKS
# -----------------------------------------------------------------------------

print("Input conservative forest years:")
print(conservative_forest_ic.aggregate_array("year").getInfo())

print("Output distance years:")
print(forest_distance_ic.aggregate_array("year").getInfo())

first_img = ee.Image(forest_distance_ic.first())
print("Example band names:", first_img.bandNames().getInfo())
print("Example projection:", first_img.projection().getInfo())

Input conservative forest years:
[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Output distance years:
[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Example band names: ['dist_from_conservative_forest_m']
Example projection: {'type': 'Projection', 'crs': 'EPSG:5070', 'transform': [30, 0, -2361600, 0, -30, 3177450]}


In [ ]:
# # -----------------------------------------------------------------------------
# # TEST: one year, Colorado visualization
# # -----------------------------------------------------------------------------

# TEST_YEAR = 2020

# # Colorado geometry
# states = ee.FeatureCollection("TIGER/2018/States")
# colorado = states.filter(ee.Filter.eq("NAME", "Colorado"))
# co_geom = colorado.geometry()

# # Grab one year of conservative forest
# forest_test = ee.Image(
#     forest_masks
#     .select("conservative")
#     .filter(ee.Filter.eq("year", TEST_YEAR))
#     .first()
# )

# # Compute distance
# dist_test = (
#     forest_test
#     .fastDistanceTransform(
#         neighborhood=DIST_NEIGHBORHOOD,
#         units="pixels",
#         metric="squared_euclidean"
#     )
#     .sqrt()
#     .multiply(forest_test.projection().nominalScale())
#     .reproject(forest_test.projection())
# )

# # Mask to Colorado for display only
# co_mask = ee.Image.constant(1).clip(co_geom).mask()
# forest_test = forest_test.updateMask(co_mask)
# dist_test = dist_test.updateMask(co_mask)

# # Quick value check
# print("Min/max distance:",
#       dist_test.reduceRegion(
#           reducer=ee.Reducer.minMax(),
#           geometry=co_geom,
#           scale=30,
#           maxPixels=1e13
#       ).getInfo())

# # Visualize
# m = geemap.Map()
# m.centerObject(colorado, 7)

# m.addLayer(forest_test, {"min": 0, "max": 1, "palette": ["white", "green"]},
#            f"Forest {TEST_YEAR}")

# m.addLayer(dist_test,
#            {"min": 0, "max": DIST_NEIGHBORHOOD * PIXEL_SIZE_M,
#             "palette": ["blue", "yellow", "red"]},
#            f"Distance {TEST_YEAR}")

# m

Min/max distance: {'distance_max': 22465.193077291813, 'distance_min': 0}


Map(center=[38.99709486991023, -105.5477142158674], controls=(WidgetControl(options=['position', 'transparent_…

In [49]:
# -----------------------------------------------------------------------------
# EXPORT EACH YEAR TO THE ASSET COLLECTION
# -----------------------------------------------------------------------------

ref_img = ee.Image(conservative_forest_ic.first())
ref_proj = ref_img.projection()
ref_crs = ref_proj.crs().getInfo()

years_to_export = forest_distance_ic.aggregate_array("year").getInfo()

for year in years_to_export:
    year = int(year)

    img = ee.Image(
        forest_distance_ic.filter(ee.Filter.eq("year", year)).first()
    )

    asset_id = f"{FOREST_DIST_IC}/dist_from_conservative_forest_{year}"

    task = ee.batch.Export.image.toAsset(
        image=img,
        description=f"dist_from_conservative_forest_{year}",
        assetId=asset_id,
        scale=EXPORT_SCALE,
        maxPixels=EXPORT_MAX_PIXELS,
        crs = ref_crs
    )

    task.start()
    print(f"Started export for year {year}: {asset_id}")

Started export for year 2000: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2000
Started export for year 2001: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2001
Started export for year 2002: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2002
Started export for year 2003: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2003
Started export for year 2004: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2004
Started export for year 2005: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2005
Started export for year 2006: projects/ee-tymc5571-multi-disturbance/assets/dist_from_conservative_forest/dist_from_conservative_forest_2006
Started expor

# Severity heterogeneity


In [ ]:
# -----------------------------------------------------------------------------
# MOVING WINDOW ON A PER-IMAGE BASIS
# -----------------------------------------------------------------------------

wumi_cbi = wumi_sev.select('CBI_bc').filter(ee.Filter.notNull(["year"])).filter(ee.Filter.inList("year", years_ee))


# CBI filled version
cbi_filled_zeros = wumi_cbi.map(
    lambda img: ee.Image(img).unmask(0).copyProperties(img, img.propertyNames())
)

def moving_window_image(
    img: ee.Image,
    window_size: int | float,
    reducer: ee.Reducer,
    kernel_shape: str = "circle",
    normalize: bool = False
) -> ee.Image:
    """
    Apply a moving window reducer to a single image and preserve properties.
    window_size is the full diameter in meters.
    """
    radius_m = window_size / 2

    if kernel_shape == "square":
        kernel = ee.Kernel.square(radius=radius_m, units="meters", normalize=normalize)
    elif kernel_shape == "circle":
        kernel = ee.Kernel.circle(radius=radius_m, units="meters", normalize=normalize)
    elif kernel_shape == "cross":
        kernel = ee.Kernel.cross(radius=radius_m, units="meters", normalize=normalize)
    elif kernel_shape == "manhattan":
        kernel = ee.Kernel.manhattan(radius=radius_m, units="meters", normalize=normalize)
    elif kernel_shape == "chebyshev":
        kernel = ee.Kernel.chebyshev(radius=radius_m, units="meters", normalize=normalize)
    else:
        raise ValueError(f"Unsupported kernel shape: {kernel_shape}")

    out = ee.Image(img).reduceNeighborhood(
        reducer=reducer,
        kernel=kernel
    )

    out = ee.Image(out).copyProperties(img, img.propertyNames())


    return out


def apply_mw_mean_std(img: ee.Image) -> ee.Image:
    img = ee.Image(img)

    mean_img = ee.Image(
        moving_window_image(
            img=img,
            window_size=MOVING_WINDOW_SIZE,
            kernel_shape="circle",
            reducer=ee.Reducer.mean(),
            normalize=False
        )
    ).rename("cbi_mw_mean")

    std_img = ee.Image(
        moving_window_image(
            img=img,
            window_size=MOVING_WINDOW_SIZE,
            kernel_shape="circle",
            reducer=ee.Reducer.stdDev(),
            normalize=False
        )
    ).rename("cbi_mw_std")

    out = ee.Image(
        mean_img.addBands(std_img)
    ).copyProperties(img, img.propertyNames()).set(
        "mw_radius", MOVING_WINDOW_SIZE / 2
    )

    return ee.Image(out)


# One image per year, with both bands
cbi_mw_ic = ee.ImageCollection(cbi_filled_zeros.map(apply_mw_mean_std))

# Optional quick checks
print("Years:", cbi_mw_ic.aggregate_array("year").getInfo())
print("Example bands:", ee.Image(cbi_mw_ic.first()).bandNames().getInfo())
print("mw_radius:", ee.Image(cbi_mw_ic.first()).get("mw_radius").getInfo())

Years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Example bands: ['cbi_mw_mean', 'cbi_mw_std']
mw_radius: 250


In [51]:
# -----------------------------------------------------------------------------
# TEST VISUALIZATION: small Colorado clip, one year
# -----------------------------------------------------------------------------

TEST_YEAR = 2020

states = ee.FeatureCollection("TIGER/2018/States")
colorado = states.filter(ee.Filter.eq("NAME", "Colorado"))

# small Front Range-ish test box; edit if you want a different area
test_geom = ee.Geometry.Rectangle(
    [-105.8, 39.9, -105.2, 40.3],
    proj=None,
    geodesic=False
)

test_img = ee.Image(
    cbi_mw_ic
    .filter(ee.Filter.eq("year", TEST_YEAR))
    .first()
).clip(test_geom)

print("Bands:", test_img.bandNames().getInfo())
print("Year:", test_img.get("year").getInfo())
print("mw_dist:", test_img.get("mw_dist").getInfo())

m = geemap.Map()
m.centerObject(test_geom, 10)

m.addLayer(
    wumi_cbi.select("CBI_bc").filter(ee.Filter.eq("year", TEST_YEAR)).first().clip(test_geom),
    {
        "min": 0,
        "max": 3,
        "palette": ["white", "khaki", "orange", "red", "darkred"]
    },
    f"CBI OG {TEST_YEAR}"
)

m.addLayer(
    test_img.select("cbi_mw_mean"),
    {
        "min": 0,
        "max": 3,
        "palette": ["white", "khaki", "orange", "red", "darkred"]
    },
    f"CBI moving window mean {TEST_YEAR}"
)

m.addLayer(
    test_img.select("cbi_mw_std"),
    {
        "min": 0,
        "max": 1.5,
        "palette": ["white", "lightblue", "blue", "purple"]
    },
    f"CBI moving window std {TEST_YEAR}"
)

m.addLayer(
    ee.Image().paint(colorado, 1, 2),
    {"palette": ["black"]},
    "Colorado outline"
)

m

Bands: ['cbi_mw_mean', 'cbi_mw_std']
Year: 2020
mw_dist: None


Map(center=[40.09993302978021, -105.4999999999996], controls=(WidgetControl(options=['position', 'transparent_…

In [52]:
# -----------------------------------------------------------------------------
# EXPORT EACH YEAR TO THE ASSET COLLECTION
# -----------------------------------------------------------------------------

ref_img = ee.Image(wumi_cbi.first())
ref_proj = ref_img.projection()
ref_crs = ref_proj.crs().getInfo()

years_to_export = cbi_mw_ic.aggregate_array("year").getInfo()

for year in years_to_export:
    year = int(year)

    img = ee.Image(
        cbi_mw_ic.filter(ee.Filter.eq("year", year)).first()
    )

    asset_id = f"{WUMI_CBI_MW_IC}/cbi_mw_{year}"

    task = ee.batch.Export.image.toAsset(
        image=img,
        description=f"cbi_mw_{year}",
        assetId=asset_id,
        scale=EXPORT_SCALE,
        maxPixels=EXPORT_MAX_PIXELS,
        crs = ref_crs
    )

    task.start()
    print(f"Started export for year {year}: {asset_id}")

Started export for year 1985: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1985
Started export for year 1986: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1986
Started export for year 1987: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1987
Started export for year 1988: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1988
Started export for year 1989: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1989
Started export for year 1990: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1990
Started export for year 1991: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1991
Started export for year 1992: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1992
Started export for year 1993: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1993
Started export for year 1994: projects/ee-tymc5571-multi-disturbance/assets/WUMI_cbi_mw/cbi_mw_1994
